# Day 12 - NLP & Classical Text Processing

## RetailMind AI Increment

Day 12 starts RetailMind AI's text-intelligence module with an auditable sentiment baseline for Amazon product reviews. The pipeline cleans noisy review text with NLTK, creates TF-IDF unigram/bigram features, trains a balanced logistic-regression classifier, evaluates several metrics, and surfaces influential positive and negative terms.

### Business objective
Review sentiment converts unstructured customer feedback into measurable product signals. The baseline can identify negative-review volume, reveal recurring quality/service terms, and establish a benchmark before Day 13 introduces BERT.

## Library and data constraints

The learning implementation uses only **scikit-learn** and **NLTK**. The dataset is supplied locally as `data/amazon_reviews.csv`; no URL is fetched by the notebook.

Accepted schemas:

```text
text,label
```

or

```text
title,content,label
```

Labels may be `0/1`, `negative/positive`, or Amazon-style `1/2`. The loader joins title and content when both are available.

In [ ]:
import sys
from pathlib import Path
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo_root / "src"))
import day12_review_sentiment_baseline as day12

print("Data path:", repo_root / day12.DATA_PATH)
print("Maximum TF-IDF features:", day12.MAX_FEATURES)

## 1. Load and inspect reviews

The holdout remains untouched until evaluation. A stratified 80/20 split keeps positive and negative proportions consistent.

In [ ]:
texts, labels = day12.load_reviews(repo_root / day12.DATA_PATH)
print("Total reviews:", len(texts))
print("Negative:", labels.count(0), "Positive:", labels.count(1))
print("Example review:", texts[0][:300])

## 2. Clean text with NLTK

The custom scikit-learn transformer performs:

1. lowercase conversion;
2. URL and HTML removal;
3. non-alphabetic noise removal;
4. NLTK word tokenization;
5. stopword filtering;
6. optional WordNet lemmatization.

A built-in fallback stopword list keeps the pipeline usable when hosted Jupyter cannot download NLTK corpora. WordNet lemmatization also degrades safely when the corpus is unavailable.

In [ ]:
cleaner = day12.ReviewCleaner().fit(texts)
for raw, cleaned in zip(texts[:3], cleaner.transform(texts[:3])):
    print("RAW   :", raw[:160])
    print("CLEAN :", cleaned[:160])
    print()

## 3. TF-IDF representation

TF-IDF assigns larger values to terms that are frequent in one review but less common across the corpus. The vectorizer uses unigrams and bigrams, removes very rare and extremely common terms, caps the vocabulary at 30,000 features, and applies sublinear term frequency. Bigrams allow phrases such as `poor quality` or `highly recommend` to carry more meaning than isolated words.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=day12.RANDOM_STATE,
    stratify=labels
)
pipeline = day12.build_pipeline()
pipeline.fit(X_train, y_train)

vectorizer = pipeline.named_steps["tfidf"]
print("Vocabulary size:", len(vectorizer.get_feature_names_out()))
print("Sample features:", vectorizer.get_feature_names_out()[:20])

## 4. Classical sentiment baseline

Logistic regression is a strong sparse-text baseline. `class_weight="balanced"` limits majority-class dominance, and the learned signed coefficients support term-level interpretation. Positive coefficients push predictions toward positive sentiment; negative coefficients push predictions toward negative sentiment.

In [ ]:
metrics, matrix, report = day12.evaluate(pipeline, X_test, y_test)
print("Confusion matrix:
", matrix)
print(report)
print("Metric summary:")
for name, value in metrics.items():
    print(f"{name:10s}: {value:.4f}")

## 5. Interpret errors honestly

Accuracy alone can hide asymmetric failures. Precision measures how often predicted positive reviews are truly positive; recall measures how many real positive reviews are captured; F1 balances both. ROC-AUC assesses ranking quality across probability thresholds. The confusion matrix exposes false-positive and false-negative counts directly.

In [ ]:
predictions = pipeline.predict(X_test)
probabilities = pipeline.predict_proba(X_test)[:, 1]

false_predictions = []
for text, truth, pred, probability in zip(X_test, y_test, predictions, probabilities):
    if truth != pred:
        false_predictions.append((truth, pred, probability, text[:220]))

print("Misclassified reviews:", len(false_predictions))
for item in false_predictions[:5]:
    print(item)

## 6. Surface informative product terms

Coefficient rankings turn the model into a basic review-insight tool. Terms should be treated as correlational signals, not causes. Product teams can use negative phrases to prioritize deeper review analysis, but sarcasm, negation, brand names, and dataset artifacts can distort rankings.

In [ ]:
positive_terms, negative_terms = day12.informative_terms(pipeline)
print("Top positive terms:")
for term, score in positive_terms[:15]:
    print(f"{term:30s} {score:+.4f}")

print("
Top negative terms:")
for term, score in negative_terms[:15]:
    print(f"{term:30s} {score:+.4f}")

## 7. Test single-review inference

In [ ]:
examples = [
    "Excellent quality and fast delivery. I would definitely buy it again.",
    "The item broke after one day and customer support was unhelpful.",
]
for review in examples:
    print(review)
    print(day12.predict_review(pipeline, review))

## 8. Persist the complete processing module

Saving the full scikit-learn pipeline keeps cleaning, TF-IDF vocabulary, feature weighting, and classifier coefficients together. In production, load the same pickle only from a trusted repository because pickle files can execute code during deserialization.

In [ ]:
import pickle

model_path = repo_root / day12.MODEL_PATH
model_path.parent.mkdir(exist_ok=True)
with open(model_path, "wb") as file:
    pickle.dump(pipeline, file)

day12.save_terms(
    positive_terms, negative_terms,
    repo_root / day12.TERMS_PATH
)
print("Saved model:", model_path)
print("Saved terms:", repo_root / day12.TERMS_PATH)

## 9. RetailMind integration and limitations

This baseline can feed product dashboards with sentiment labels, confidence scores, and frequent influential terms. Key limitations are bag-of-words context loss, weak handling of sarcasm and long-distance negation, domain shift between product categories, and possible label noise. Day 13 should compare BERT against this fixed test split rather than replacing the baseline without measurement.